In [111]:
# Standard libraries
import os
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, mutual_info_regression, f_classif, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Other utilities
from scipy.stats import randint, uniform
import joblib

# Standard libraries
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, f_classif, f_regression

In [112]:
path = "../../data/raw/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

Loaded 01_DiatomInventories_GTstudentproject_B with shape (1643872, 8)
Loaded 02_InfoSites_GTstudentproject_B with shape (8404, 11)
Loaded 03_IBD_GTstudentproject_test with shape (5063, 2)
Loaded 03_IBD_GTstudentproject_train with shape (43783, 4)
Loaded 04_PressureStatus_GTstudentproject_B with shape (49231, 30)
Loaded 05_EnvParamMeans_GTstudentproject_B with shape (3763903, 8)
Loaded 06_ListEnvParam_GNNprojectGT_B with shape (192, 14)
Loaded 07_TaxaCode_GTstudentproject_B with shape (2292, 2)


In [113]:
taxones = dfs[list(dfs.keys())[0]] 

In [114]:
pressure = dfs[list(dfs.keys())[4]]

In [115]:
epm = dfs[list(dfs.keys())[5]]

In [116]:
path = "../../data/processed/"
dfs_processed = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs_processed[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs_processed[name].shape}")

Loaded 03_CLEAN_COMPLETE_DF with shape (49863, 174)
Loaded 03_CLEAN_COMPLETE_DF_02 with shape (49231, 623)
Loaded 03_COMPLETE_TEST with shape (43568, 3)
Loaded 03_COMPLETE_TRAIN with shape (49441, 2332)
Loaded 03_COMPLETE_TRAIN_2 with shape (49231, 2849)
Loaded clean_train with shape (43568, 4)
Loaded dep_codes with shape (49231, 12)
Loaded dep_test with shape (5063, 12)
Loaded taxones_pressure with shape (5663, 2333)
Loaded taxones_pressure_epm_predict with shape (5663, 2850)
Loaded taxones_pressure_epm_train with shape (43568, 2853)
Loaded taxones_pressure_predict with shape (5663, 2333)
Loaded taxones_pressure_train with shape (43568, 2336)


In [117]:
sites = dfs_processed[list(dfs_processed.keys())[6]]

In [118]:
train = dfs_processed[list(dfs_processed.keys())[5]]

In [119]:
test = dfs_processed[list(dfs_processed.keys())[7]]

In [120]:
test_codes = test['CodeDepartement'].unique()

In [121]:
regiones = sites['HERlvl1Code'].drop_duplicates().tolist()
len(regiones)

22

ejemplo antes de volverlo función

In [134]:
region = 21

df = sites[sites['HERlvl1Code']== region]

valid_codes = df ['SamplingOperations_code'].unique()

df1 = taxones[taxones['SamplingOperations_code'].isin(valid_codes)]

# 1) columnas que QUIERES mantener en la llave (todas menos: TaxonName y Abundance_nbcell;
#    y quitamos TaxonCode/Abundance_pm porque se usan para columnas/valores)
cols_key = [c for c in df1.columns 
            if c not in ['TaxonName', 'Abundance_nbcell', 'TaxonCode', 'Abundance_pm']]

# 2) tabla pivote:
#    - index = todas las columnas que quieres conservar (parte del key)
#    - columns = códigos de taxón
#    - values = Abundance_pm
#    - aggfunc='sum' por si hay duplicados del mismo taxón en la misma muestra
wide = (
    df1.pivot_table(index=cols_key,
                   columns='TaxonCode',
                   values='Abundance_pm',
                   aggfunc='sum')
      .reset_index()
)

# opcional: quitar el nombre del eje de columnas
wide.columns.name = None

df2 = wide

df2 = pd.merge(wide, pressure, on=['SamplingOperations_code', 'CodeSite_SamplingOperations','Date_SamplingOperation'],how = 'inner')

train_df = pd.merge(df2, train, on= 'SamplingOperations_code',how = 'inner')
test_df = df2[df2['SamplingOperations_code'].isin(test_codes)]


In [136]:
def drop_exact_duplicates(X: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop exact duplicate columns from a DataFrame.

    This function identifies and removes columns in the DataFrame that are exact duplicates 
    of other columns. Duplicate columns are those that have identical values across all rows.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with duplicate columns removed.
        - A list of the names of the dropped duplicate columns.

    Notes:
    -----
    - The function computes a hash signature for each column to efficiently identify duplicates.
    - If two columns have the same hash and their values are identical, one of them is dropped.

    Example Usage:
    --------------
    X, dropped_dup_cols = drop_exact_duplicates(X)
    print(f"Dropped exact duplicate columns: {dropped_dup_cols}")
    """
    sig = X.apply(lambda s: pd.util.hash_pandas_object(s, index=False).sum())
    seen = {}
    dup = []
    for c, h in sig.items():
        if h in seen and X[c].equals(X[seen[h]]):
            dup.append(c)
        else:
            seen[h] = c
    return X.drop(columns=dup), dup

def drop_high_missing(X: pd.DataFrame, thresh: float = 0.40) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop columns with a high proportion of missing values from a DataFrame.

    This function identifies columns in the DataFrame where the proportion of missing values 
    exceeds a specified threshold and removes them. Missing values are identified as NaN 
    or the string "Unassessed", which is replaced with NaN before processing.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    thresh : float, optional
        The proportion threshold above which a column is considered to have high missing values 
        and is dropped. Default is 0.40 (40%).

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with high-missing columns removed.
        - A list of the names of the dropped columns.

    Notes:
    -----
    - The function replaces the string "Unassessed" with NaN before calculating the proportion 
      of missing values.
    - Columns with a proportion of missing values greater than `thresh` are dropped.

    Example Usage:
    --------------
    X, dropped_cols = drop_high_missing(X, thresh=0.40)
    print(f"Dropped columns with >40% missing: {dropped_cols}")
    """
    # Replace "Unassessed" with NaN
    X = X.replace("Unassessed", np.nan)

    # Identify columns with a high proportion of missing values
    to_drop = X.columns[X.isna().mean() > thresh].tolist()

    # Drop the identified columns
    X2 = X.drop(columns=to_drop)

    return X2, to_drop

def drop_quasi_constant_cat(
    X: pd.DataFrame, 
    cat_cols: List[str], 
    p: float = 0.99
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop quasi-constant categorical columns from a DataFrame.

    This function identifies categorical columns where the most frequent value 
    accounts for at least `p` proportion of the data and removes them from the DataFrame. 
    Quasi-constant columns are those that provide little variability and are unlikely 
    to be useful for modeling.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    cat_cols : List[str]
        A list of categorical column names to consider for quasi-constant checking.
    p : float, optional
        The proportion threshold above which a column is considered quasi-constant.
        Default is 0.99.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with quasi-constant categorical columns removed.
        - A list of the names of the dropped quasi-constant categorical columns.

    Notes:
    -----
    - The function ensures that only columns present in the DataFrame are processed.
    - Missing values are included in the value counts when determining the most frequent value.

    Example Usage:
    --------------
    cat_cols = X.select_dtypes(exclude=['number']).columns
    X, dropped_cat_cols = drop_quasi_constant_cat(X, cat_cols.tolist(), p=0.99)
    print(f"Dropped quasi-constant categorical columns: {dropped_cat_cols}")
    """
    dropped = []
    for c in cat_cols:
        # Calculate the normalized value counts (including NaNs)
        vc = X[c].value_counts(normalize=True, dropna=False)
        
        # Check if the most frequent value exceeds the threshold
        if len(vc) and vc.iloc[0] >= p:
            dropped.append(c)
    
    # Drop the identified quasi-constant columns
    X2 = X.drop(columns=dropped)
    return X2, dropped

def drop_high_cardinality_cat(
    X: pd.DataFrame, 
    cat_cols: List[str], 
    max_unique: int = 100, 
    ratio: float = 0.50
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop high-cardinality categorical columns from a DataFrame.

    This function identifies categorical columns with a high number of unique values 
    (cardinality) and removes them from the DataFrame. High-cardinality columns can 
    increase the complexity of the model and may not provide significant value.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    cat_cols : List[str]
        A list of categorical column names to consider for cardinality checking.
    max_unique : int, optional
        The maximum number of unique values allowed in a categorical column. 
        Default is 100.
    ratio : float, optional
        The maximum ratio of unique values to the total number of rows in the DataFrame.
        Default is 0.50.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with high-cardinality categorical columns removed.
        - A list of the names of the dropped high-cardinality categorical columns.

    Notes:
    -----
    - A column is considered high-cardinality if the number of unique values exceeds 
      `max_unique` or if the ratio of unique values to the total number of rows exceeds `ratio`.
    - The function ensures that only columns present in the DataFrame are processed.

    Example Usage:
    --------------
    cat_cols = X.select_dtypes(exclude=['number']).columns
    X, dropped_high_card_cols = drop_high_cardinality_cat(X, cat_cols.tolist(), max_unique=100, ratio=0.50)
    print(f"Dropped high-cardinality categorical columns: {dropped_high_card_cols}")
    """
    n = len(X)
    lim = min(max_unique, int(ratio * n))
    dropped = [c for c in cat_cols if c in X.columns and X[c].nunique(dropna=False) > lim]
    X2 = X.drop(columns=dropped)
    return X2, dropped

def drop_quasi_constant_num(
    X: pd.DataFrame, num_cols: List[str], thresh: float = 1e-5
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop quasi-constant numeric columns from a DataFrame.

    This function identifies numeric columns with very low variance (below a specified threshold)
    and removes them from the DataFrame. Quasi-constant columns are those that have almost the same
    value across all rows, which makes them uninformative for modeling.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    num_cols : List[str]
        A list of numeric column names to consider for variance checking.
    thresh : float, optional
        The variance threshold below which columns are considered quasi-constant.
        Default is 1e-5.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with quasi-constant numeric columns removed.
        - A list of the names of the dropped quasi-constant numeric columns.

    Notes:
    -----
    - Missing values in the numeric columns are imputed using the median before calculating variance.
    - The VarianceThreshold from sklearn is used to identify quasi-constant columns.
    """
    if not num_cols:
        return X, []

    # Impute missing values with the median
    imp = SimpleImputer(strategy="median")
    Xn = pd.DataFrame(imp.fit_transform(X[num_cols]), columns=num_cols, index=X.index)

    # Apply VarianceThreshold to identify columns with low variance
    vt = VarianceThreshold(threshold=thresh)
    vt.fit(Xn)

    # Identify kept and dropped columns
    kept = [c for c, k in zip(num_cols, vt.get_support()) if k]
    dropped = [c for c in num_cols if c not in kept]

    # Drop the quasi-constant columns from the original DataFrame
    X2 = X.drop(columns=dropped)
    return X2, dropped

def prefilter_num_univariate(
    X: pd.DataFrame, y: pd.Series, num_cols: List[str], k: int = 300
) -> Tuple[pd.DataFrame, List[str], pd.Series]:
    """
    Perform a univariate feature selection for numeric columns based on their relationship with the target variable.

    This function selects the top `k` numeric features that have the highest scores in a univariate statistical test 
    (ANOVA F-value for classification or F-statistic for regression) with respect to the target variable `y`.

    Parameters:
    ----------
    X : pd.DataFrame
        The input dataframe containing features.
    y : pd.Series
        The target variable.
    num_cols : List[str]
        A list of numeric column names to consider for feature selection.
    k : int, optional
        The number of top features to keep, by default 300.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str], pd.Series]
        - A dataframe containing the top `k` numeric features.
        - A list of the names of the selected top `k` numeric features.
        - A pandas Series containing the scores of all numeric features, sorted in descending order.

    Notes:
    -----
    - If the target variable `y` is numeric and has more than 20 unique values, the function uses `f_regression`.
      Otherwise, it uses `f_classif`.
    - Missing values in the numeric columns are imputed using the median before calculating the scores.
    """
    # Filter numeric columns that exist in the dataframe
    keep = [c for c in num_cols if c in X.columns]
    if not keep:
        return X, [], pd.Series(dtype=float)

    # Impute missing values with the median
    Xi = pd.DataFrame(
        SimpleImputer(strategy="median").fit_transform(X[keep]),
        columns=keep, index=X.index
    )

    # Select the appropriate statistical test based on the target variable type
    if np.issubdtype(y.dtype, np.number) and y.nunique() > 20:
        scores, _ = f_regression(Xi, y.to_numpy())
    else:
        scores, _ = f_classif(Xi, y.to_numpy())

    # Create a pandas Series of scores, sort them in descending order
    s = pd.Series(scores, index=keep).fillna(0.0).sort_values(ascending=False)

    # Select the top `k` features
    top = s.index[: min(k, len(s))].tolist()

    # Return the top features, their names, and the scores
    return pd.concat([Xi[top]], axis=1), top, s

def drop_zeros(X: pd.DataFrame, thresh: float = 0.95) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop columns with a high proportion of zero values from a DataFrame.

    This function identifies columns in the DataFrame where the proportion of zero values 
    exceeds a specified threshold and removes them. Columns with a high proportion of zeros 
    are often uninformative and can be excluded from further analysis.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    thresh : float, optional
        The proportion threshold above which a column is considered to have high zero values 
        and is dropped. Default is 0.95 (95%).

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with high-zero columns removed.
        - A list of the names of the dropped columns.

    Notes:
    -----
    - The function calculates the proportion of zero values in each column.
    - Columns with a proportion of zero values greater than `thresh` are dropped.

    Example Usage:
    --------------
    Xnum, dropped_cols = drop_zeros(Xnum, thresh=0.95)
    print(f"Dropped columns with >95% zeros: {dropped_cols}")
    """
    # Identify columns with a high proportion of zero values
    to_drop = X.columns[(X == 0).mean() > thresh].tolist()

    # Drop the identified columns
    X = X.drop(columns=to_drop)

    return X, to_drop


In [137]:
def clean_up(train,test):
    train.set_index('SamplingOperations_code')
    Xtr = train.drop(columns=(['IBD', 'IBD_EQR', 'IBD_EQR_Status'])).set_index('SamplingOperations_code')
    Y = train[['SamplingOperations_code', 'IBD', 'IBD_EQR', 'IBD_EQR_Status']].set_index('SamplingOperations_code')
    y = train[['IBD']]
    Xte=test.set_index('SamplingOperations_code')
    
    X = pd.concat([Xtr, Xte], axis=0)
    COMPLETE_X = X.copy()
    X = X.drop(columns=['Date_SamplingOperation'])

    """En este espacio es dodne podemos hacer limpieza de datos """
    X, dropped_dup_cols = drop_exact_duplicates(X)
    print(f"Dropped exact duplicate columns: {dropped_dup_cols}")

    X, dropped_cols = drop_high_missing(X, thresh=0.95)
    print(f"Dropped columns with >95% missing: {dropped_cols}")
    print(len(dropped_cols))

    cat_cols = X.select_dtypes(exclude=['number']).columns
    X, dropped_cat_cols = drop_quasi_constant_cat(X, cat_cols.tolist(), p=0.99)
    print(f"Dropped quasi-constant categorical columns: {dropped_cat_cols}")

    cat_cols = X.select_dtypes(exclude=['number']).columns
    X, dropped_high_card_cols = drop_high_cardinality_cat(X, cat_cols.tolist(), max_unique=100, ratio=0.50)
    print(f"Dropped high-cardinality categorical columns: {dropped_high_card_cols}")

    num_cols = X.select_dtypes(include=['number']).columns  # Select numeric columns
    X, dropped_num_cols = drop_quasi_constant_num(X, num_cols.tolist(), thresh=1e-5)
    print(f"Dropped quasi-constant numeric columns: {dropped_num_cols}")
    
    COMPLETE_X[dropped_cols]

    uncommon_taxons = COMPLETE_X[dropped_cols].sum(axis=1)
    uncommon_taxons = uncommon_taxons.rename("Uncommon_Taxons")

    df3 = pd.concat([X, uncommon_taxons], axis=1)

    df3 = df3.join(Y, how='left')

    

    return df3 

In [125]:
def get_region_sites_and_taxa(region, sites, taxones):
    """
    Regresa los dos DF que armaste a mano:
    - df: subset de sites para la región
    - df1: subset de taxones filtrado por los SamplingOperations_code válidos de df
    """
    df = sites[sites['HERlvl1Code'] == region].copy()
    valid_codes = df['SamplingOperations_code'].unique()
    df1 = taxones[taxones['SamplingOperations_code'].isin(valid_codes)].copy()
    return  df1


In [126]:
def get_region_sites_and_epm(region, sites, epm):
    """
    Regresa los dos DF que armaste a mano:
    - df: subset de sites para la región
    - df1: subset de taxones filtrado por los SamplingOperations_code válidos de df
    """
    df = sites[sites['HERlvl1Code'] == region].copy()
    valid_codes = df['SamplingOperations_code'].unique()
    dfe = epm[epm['SamplingOperations_code'].isin(valid_codes)].copy()
    return  dfe

In [127]:
def pivot_taxa_abundance_pm(df1):
    """
    Hace el pivote exactamente como en tu código:
    - index: todas las columnas salvo TaxonName, Abundance_nbcell, TaxonCode, Abundance_pm
    - columns: TaxonCode
    - values: Abundance_pm
    - aggfunc: sum
    """
    # columnas que conservas en la llave
    drop_cols = ['TaxonName', 'Abundance_nbcell', 'TaxonCode', 'Abundance_pm']
    cols_key = [c for c in df1.columns if c not in drop_cols]

    wide = (
        df1.pivot_table(index=cols_key,
                        columns='TaxonCode',
                        values='Abundance_pm',
                        aggfunc='sum')
           .reset_index()
    )
    wide.columns.name = None
    return wide


In [128]:
def pivot_pressure_means(dfe, agg='first'):
    """
    Pivotea un df de presión para obtener columnas por código de parámetro y
    valores = Mean1Y / Mean180Days / Mean90Days.

    Parámetros
    ----------
    df : DataFrame
        Debe contener: 
        ['CodeSite_SamplingOperations', 'SamplingOperations_code', 
         'Parametre_code', 'Mean1Y', 'Mean180Days', 'Mean90Days'].
    agg : str or callable, default 'first'
        Función de agregación cuando hay duplicados por (keys, Parametre_code).
        Ejemplos: 'first', 'mean', 'max', np.mean, etc.
    drop_code_site : bool, default True
        Si True, elimina la columna 'CodeSite_SamplingOperations' al final.

    Return
    ------
    wide2 : DataFrame
        DataFrame ancho con columnas como:
        Mean1Y_XXXX, Mean180Days_XXXX, Mean90Days_XXXX (XXXX = Parametre_code),
        e índice reseteado.
    """
    # Claves del índice
    keys = ['CodeSite_SamplingOperations', 'SamplingOperations_code']

    # Pivot con columnas multiíndice (stat, Parametre_code)
    wide = (
        dfe.pivot_table(
            index=keys,
            columns='Parametre_code',
            values=['Mean1Y', 'Mean180Days', 'Mean90Days'],
            aggfunc=agg
        )
        .sort_index(axis=1)  # orden estable
    )

    # Aplana nombres de columnas -> Mean1Y_1319, Mean180Days_1319, ...
    wide.columns = [f"{stat}_{code}" for stat, code in wide.columns]

    # Resetea índice para volver a columnas normales
    wide2 = wide.reset_index()


    return wide2


In [129]:
def build_region_train_test(region, sites, taxones, pressure, train, test_codes, fillna_with=0):
    """
    Flujo completo por región:
    1) Filtra sites y taxones (df, df1)
    2) Pivotea abundancias permil por TaxonCode (wide)
    3) Une con pressure (df2) usando las mismas llaves que tú pusiste
    4) Saca train_df (inner contra 'train' por SamplingOperations_code)
    5) Saca test_df filtrando df2 por test_codes

    Parámetros:
      - region: int/str con el código de región (HERlvl1Code)
      - sites, taxones, pressure, train: DataFrames existentes
      - test_codes: iterable de SamplingOperations_code para test
      - fillna_with: si quieres rellenar NaN post-pivote (0 por default). Usa None para no rellenar.

    Regresa:
      - train_df, test_df
    """
    # 1) df y df1
    df1 = get_region_sites_and_taxa(region, sites, taxones)

    # Si no hay datos para la región, devuelve vacíos con mismas columnas.
    if df1.empty:
        empty_train = pd.DataFrame(columns=['SamplingOperations_code'])
        empty_test  = pd.DataFrame(columns=['SamplingOperations_code'])
        return empty_train, empty_test

    # 2) pivote
    wide = pivot_taxa_abundance_pm(df1)

    # 3) merge con pressure (exactamente tus llaves)
    df2 = pd.merge(
        wide,
        pressure,
        on=['SamplingOperations_code', 'CodeSite_SamplingOperations', 'Date_SamplingOperation'],
        how='inner'
    )

   
    df3 = pd.merge(df2, train, on='SamplingOperations_code', how='left')    
    # 5) test_df (filtrado por test_codes)
    col = 'IBD'   # columna en dfB que checas

    mask = df3[col].isna()          # True si NaN

    train_df = df3.loc[~mask]
    test_df = df3.loc[mask]  
    test_df=test_df.drop(columns=(['IBD', 'IBD_EQR', 'IBD_EQR_Status']))

    cleandf = clean_up(train= train_df,test=test_df)

    return cleandf


In [130]:
# Ejemplo de uso:
region = 21
cleandf= build_region_train_test(
    region=region,
    sites=sites,
    taxones=taxones,
    pressure=pressure,
    train=train,
    test_codes=test_codes,   # asegúrate de tenerlo definido
    fillna_with=0            # pon None si no quieres rellenar
)


Dropped exact duplicate columns: ['Pinrh02']
Dropped columns with >95% missing: ['Achaa01', 'Achac01', 'Achaf01', 'Achaf02', 'Achal01', 'Achan01', 'Achba01', 'Achbi02', 'Achca03', 'Achca04', 'Achco01', 'Achco02', 'Achcr01', 'Achde02', 'Achde03', 'Achdi01', 'Achdi02', 'Achdr01', 'Achel01', 'Achen01', 'Achex01', 'Achex02', 'Achfr01', 'Achfu01', 'Achge01', 'Achgr02', 'Achgr03', 'Achhe01', 'Achho01', 'Achhu01', 'Achja02', 'Achjo01', 'Achko01', 'Achku01', 'Achla03', 'Achla04', 'Achla06', 'Achle02', 'Achli01', 'Achli02', 'Achli03', 'Achlu02', 'Achma01', 'Achmi03', 'Achna02', 'Achne01', 'Achno01', 'Achpa01', 'Achpe02', 'Achpf01', 'Achps04', 'Achpu01', 'Achpy01', 'Achre01', 'Achro01', 'Achro02', 'Achru01', 'Achsa01', 'Achse02', 'Achsi01', 'Achst01', 'Achst03', 'Achsu06', 'Achsu07', 'Achte01', 'Achtr02', 'Achtr03', 'Achzh01', 'Achzi01', 'Actde01', 'Actno01', 'Adlba01', 'Adlbr01', 'Adlbr02', 'Adlmu02', 'Adlpa01', 'Adlsu01', 'Ampat01', 'Ampei01', 'Ampma01', 'Ampme01', 'Ampmi02', 'Ampmo01', 'Ampov

In [131]:
""""import os

# Definir el path base
base_path = r"dfs_taxon_p_epm"

# Crear la carpeta si no existe
os.makedirs(base_path, exist_ok=True)

# Loop por regiones y guardar cada df
for region in regiones:
    cleandf = build_region_train_test(
        region=region,
        sites=sites,
        taxones=taxones,
        pressure=pressure,
        train=train,
        test_codes=test_codes,
        fillna_with=0
    )

    # Nombre del archivo, por ejemplo df_PACA.parquet
    file_name = f"df_{region}.parquet"
    file_path = os.path.join(base_path, file_name)

    # Guardar el DataFrame
    cleandf.to_parquet(file_path, index=False)
    print(f"Guardado: {file_path}")

""""


SyntaxError: unterminated string literal (detected at line 29) (2757406436.py, line 29)

In [138]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# =========================
# Helpers: componentes conexas
# =========================
def _connected_components_from_adj(adj_df: pd.DataFrame) -> list[list[str]]:
    nodes = adj_df.index.tolist()
    A = adj_df.values.astype(bool)
    visited = set()
    comps = []
    for i in range(len(nodes)):
        if nodes[i] in visited:
            continue
        stack = [i]
        comp_idx = set()
        while stack:
            u = stack.pop()
            if u in comp_idx:
                continue
            comp_idx.add(u)
            neigh = np.where(A[u])[0]
            for w in neigh:
                if w not in comp_idx:
                    stack.append(w)
        for j in comp_idx:
            visited.add(nodes[j])
        comps.append([nodes[j] for j in comp_idx])
    return comps

# =========================
# NUMÉRICAS: correlación robusta
# =========================
def build_robust_corr(
    df: pd.DataFrame,
    num_cols=None,
    method: str = "pearson",
    min_pairs: int = 50,
    min_pair_coverage: float = 0.30,
    drop_almost_constant: bool = True,
    const_std_tol: float = 1e-12,
    min_col_coverage: float = 0.50
):
    if num_cols is None:
        num_cols = df.select_dtypes(include=["number"]).columns.tolist()
    num_df = df[num_cols].copy().replace([np.inf, -np.inf], np.nan)
    N = len(num_df)

    dropped = {"low_coverage": [], "constant": []}

    col_cov = num_df.notna().mean(axis=0)
    low_cov_cols = col_cov[col_cov < min_col_coverage].index.tolist()
    if low_cov_cols:
        num_df = num_df.drop(columns=low_cov_cols)
        dropped["low_coverage"].extend(low_cov_cols)

    if drop_almost_constant and num_df.shape[1] > 0:
        stds = num_df.std(skipna=True)
        nunq = num_df.nunique(dropna=True)
        const_cols = list(set(stds[stds <= const_std_tol].index.tolist() + nunq[nunq <= 1].index.tolist()))
        if const_cols:
            num_df = num_df.drop(columns=const_cols)
            dropped["constant"].extend(const_cols)

    kept_cols = num_df.columns.tolist()
    if not kept_cols:
        return (pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), kept_cols, dropped)

    valid = num_df.notna().astype(int)
    pair_counts = pd.DataFrame(valid.T.values @ valid.values, index=kept_cols, columns=kept_cols)
    pair_cov = pair_counts / float(N)

    corr = num_df.corr(method=method)
    low_ev = (pair_counts < min_pairs) | (pair_cov < min_pair_coverage)
    corr = corr.mask(low_ev)

    return corr, pair_counts, pair_cov, kept_cols, dropped

def drop_high_corr_numeric(
    X_num: pd.DataFrame,
    y: pd.Series,
    corr_method="spearman",
    corr_thr=0.95,
    min_pairs=5000,
    min_pair_coverage=0.60,
    min_col_coverage=0.70
):
    corr, _, _, kept_cols, dropped0 = build_robust_corr(
        X_num,
        num_cols=X_num.columns.tolist(),
        method=corr_method,
        min_pairs=min_pairs,
        min_pair_coverage=min_pair_coverage,
        drop_almost_constant=True,
        const_std_tol=1e-12,
        min_col_coverage=min_col_coverage
    )
    if corr.empty or not kept_cols:
        return kept_cols, [], dropped0

    # correlación con el target (absoluta)
    xy = X_num[kept_cols].join(y.rename("y")).dropna()
    if xy.empty:
        # si no hay pares válidos con y, no forzamos drops adicionales
        return kept_cols, [], dropped0

    target_corr = xy[kept_cols].corrwith(xy["y"], method=corr_method).abs().to_dict()

    # grafo de alta correlación |r| >= thr
    adj = corr.abs() >= corr_thr
    adj = adj.fillna(False)
    np.fill_diagonal(adj.values, False)

    comps = _connected_components_from_adj(adj)
    keep, drop = set(), set()
    for comp in comps:
        if len(comp) == 1:
            keep.add(comp[0])
            continue
        best = max(comp, key=lambda c: (target_corr.get(c, np.nan), ))
        keep.add(best)
        drop.update([c for c in comp if c != best])

    # columnas sin edges
    all_in_groups = set(sum(comps, []))
    isolated = set(kept_cols) - all_in_groups
    keep.update(isolated)

    kept_final = sorted(list(keep))
    dropped_num = sorted(list(drop))
    dropped_info = {"num_lowcov_or_constant": dropped0["low_coverage"] + dropped0["constant"],
                    "num_redundant": dropped_num}
    return kept_final, dropped_num, dropped_info

# =========================
# CATEGÓRICAS: asociación & redundancia
# =========================
def eta_squared_by_category(y: pd.Series, x: pd.Series) -> float:
    df_ = pd.DataFrame({'y': y, 'x': x}).dropna()
    if df_.empty:
        return np.nan
    overall = df_['y'].mean()
    grp = df_.groupby('x')['y'].agg(['count', 'mean'])
    ssb = (grp['count'] * (grp['mean'] - overall) ** 2).sum()
    sst = ((df_['y'] - overall) ** 2).sum()
    return float(ssb / sst) if sst > 0 else np.nan

def cramers_v_corrected(a: pd.Series, b: pd.Series) -> float:
    df_ = pd.DataFrame({'a': a, 'b': b}).dropna()
    n = len(df_)
    if n == 0:
        return np.nan
    tbl = pd.crosstab(df_['a'], df_['b'])
    r, k = tbl.shape
    if r < 2 or k < 2:
        return np.nan
    chi2, _, _, _ = chi2_contingency(tbl, correction=False)
    phi2 = chi2 / n
    phi2corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / max(n - 1, 1))
    rcorr = r - ((r - 1) ** 2) / max(n - 1, 1)
    kcorr = k - ((k - 1) ** 2) / max(n - 1, 1)
    denom = max(min(rcorr - 1, kcorr - 1), 1e-12)
    v = np.sqrt(phi2corr / denom)
    return float(np.clip(v, 0.0, 1.0))

def drop_redundant_categoricals(
    X_cat: pd.DataFrame,
    y: pd.Series,
    min_col_coverage=0.70,
    dominance_thr=0.95,
    max_levels=200,
    min_pair_cover=0.60,
    min_pair_count=5000,
    cramers_thr=0.85
):
    cat_df = X_cat.replace([np.inf, -np.inf], np.nan)
    N = len(cat_df)

    # métricas por columna
    info = []
    for c in cat_df.columns:
        s = cat_df[c]
        cov = s.notna().mean()
        nun = s.nunique(dropna=True)
        dom = (s.value_counts(dropna=True, normalize=True).iloc[0] if nun > 0 else np.nan)
        info.append({'col': c, 'coverage': cov, 'n_levels': nun, 'dominance': dom})
    smry = pd.DataFrame(info)

    low_cov = smry.loc[smry.coverage < min_col_coverage, 'col'].tolist()
    dom_cols = smry.loc[smry.dominance >= dominance_thr, 'col'].tolist()
    one_lvl = smry.loc[smry.n_levels <= 1, 'col'].tolist()

    candidates = smry.loc[
        (smry.coverage >= min_col_coverage) & (smry.n_levels > 1),
        'col'
    ].tolist()

    # eta2 con target
    eta2_map = {c: eta_squared_by_category(y, cat_df[c]) for c in candidates}

    # limitar cardinalidad antes de V
    levels_map = cat_df[candidates].nunique(dropna=True).to_dict()
    small_card = [c for c in candidates if levels_map.get(c, 0) <= max_levels]

    # matrices de evidencia
    V_index = small_card
    notna_mat = cat_df[V_index].notna().astype(int)
    pair_counts = pd.DataFrame(notna_mat.T.values @ notna_mat.values, index=V_index, columns=V_index)
    pair_cov = pair_counts / float(N)

    # Cramer's V con filtros
    V = pd.DataFrame(index=V_index, columns=V_index, dtype=float)
    for i, c1 in enumerate(V_index):
        V.loc[c1, c1] = 1.0
        for j in range(i+1, len(V_index)):
            c2 = V_index[j]
            if (pair_counts.loc[c1, c2] < min_pair_count) or (pair_cov.loc[c1, c2] < min_pair_cover):
                v = np.nan
            else:
                v = cramers_v_corrected(cat_df[c1], cat_df[c2])
            V.loc[c1, c2] = v
            V.loc[c2, c1] = v

    # grupos redundantes
    adj = (V >= cramers_thr).fillna(False)
    np.fill_diagonal(adj.values, False)
    comps = _connected_components_from_adj(adj)

    keep, drop = set(), set()
    for comp in comps:
        if len(comp) == 1:
            keep.add(comp[0]); continue
        best = max(comp, key=lambda c: (eta2_map.get(c, np.nan), -levels_map.get(c, 10**9)))
        keep.add(best)
        drop.update([c for c in comp if c != best])

    all_in_groups = set(sum(comps, []))
    isolated = set(V_index) - all_in_groups
    keep.update(isolated)

    cat_keep = sorted(list(keep))
    cat_drop = sorted(list(drop))
    discard_all = sorted(set(low_cov + dom_cols + one_lvl + cat_drop))

    dropped_info = {
        "cat_lowcov": low_cov,
        "cat_dominant": dom_cols,
        "cat_one_level": one_lvl,
        "cat_redundant": cat_drop,
        "cat_total_dropped": discard_all
    }
    return cat_keep, discard_all, dropped_info


In [139]:

# =========================
# WRAPPER PRINCIPAL: cleaner
# =========================
def cleaner(cleandf: pd.DataFrame):
    target_cols=('IBD','IBD_EQR','IBD_EQR_Status'),
    index_col='SamplingOperations_code',
    corr_thr_num=0.95,
    corr_method_num='spearman',
    min_pairs=5000,
    min_pair_cov=0.60,
    min_col_cov=0.70,
    cramers_thr=0.85

    """
    Segundo 'clean' (después de clean_up):
    - Solo elimina redundancia entre features (num & cat).
    - Mantiene la feature del grupo más asociada al objetivo.
    - Devuelve DF con el MISMO índice y targets intactos.

    Returns
    -------
    cleaned : DataFrame
    kept_cols : list (features conservadas, sin targets)
    dropped_info : dict
    """
    # separar TRAIN para medir asociación con el target
    has_y = cleandf[target_cols[0]].notna()
    TRAIN = cleandf.loc[has_y].copy()

    # X/y para métricas
    y = TRAIN[target_cols[0]]
    X = TRAIN.drop(columns=[c for c in target_cols if c in TRAIN.columns], errors='ignore')

    # dividir por tipo
    num_cols = X.select_dtypes(include=['number']).columns.tolist()
    cat_cols = X.columns.difference(num_cols).tolist()

    # --- Numéricas
    num_keep, _, num_dropinfo = drop_high_corr_numeric(
        TRAIN[num_cols],
        y,
        corr_method=corr_method_num,
        corr_thr=corr_thr_num,
        min_pairs=min_pairs,
        min_pair_coverage=min_pair_cov,
        min_col_coverage=min_col_cov
    )

    # --- Categóricas
    if cat_cols:
        cat_keep, _, cat_dropinfo = drop_redundant_categoricals(
            TRAIN[cat_cols],
            y,
            min_col_coverage=min_col_cov,
            dominance_thr=0.95,
            max_levels=200,
            min_pair_cover=min_pair_cov,
            min_pair_count=min_pairs,
            cramers_thr=cramers_thr
        )
    else:
        cat_keep, cat_dropinfo = [], {"cat_total_dropped": []}

    kept_cols = sorted(list(set(num_keep + cat_keep)))

    # construir DF final: (solo quitamos las columnas descartadas)
    cols_to_keep = [index_col] + kept_cols + [c for c in target_cols if c in cleandf.columns]
    cols_to_keep = [c for c in cols_to_keep if c in cleandf.columns]  # robustez

    cleaned = cleandf[cols_to_keep].copy().set_index(index_col)

    dropped_info = {
        **num_dropinfo,
        **cat_dropinfo
    }
    return cleaned

In [141]:
def build_region_train_test_epm(region, sites, taxones, pressure, train, test_codes, fillna_with=0):
    """
    Flujo completo por región:
    1) Filtra sites y epm (df, df1)
    1.5) Filtra sites y taxones (df, dfe)
    2) Pivotea abundancias permil por TaxonCode (wide)
    3) Une con pressure (df2) usando las mismas llaves que tú pusiste
    4) Saca train_df (inner contra 'train' por SamplingOperations_code)
    5) Saca test_df filtrando df2 por test_codes

    Parámetros:
      - region: int/str con el código de región (HERlvl1Code)
      - sites, taxones, pressure, train: DataFrames existentes
      - test_codes: iterable de SamplingOperations_code para test
      - fillna_with: si quieres rellenar NaN post-pivote (0 por default). Usa None para no rellenar.

    Regresa:
      - train_df, test_df
    """
    # 1) df y df1
    df1 = get_region_sites_and_taxa(region, sites, taxones)
    
    dfe = get_region_sites_and_epm(region, sites, epm)

    # Si no hay datos para la región, devuelve vacíos con mismas columnas.
    if df1.empty:
        empty_train = pd.DataFrame(columns=['SamplingOperations_code'])
        empty_test  = pd.DataFrame(columns=['SamplingOperations_code'])
        return empty_train, empty_test

    # 2) pivote
    wide = pivot_taxa_abundance_pm(df1)
    wide2=pivot_pressure_means(dfe)

    # 3) merge con pressure (exactamente tus llaves)
    df2 = pd.merge(
        wide,
        pressure,
        on=['SamplingOperations_code', 'CodeSite_SamplingOperations', 'Date_SamplingOperation'],
        how='inner'
    )

    df2 = pd.merge(
        df2,
        wide2,
        on=['SamplingOperations_code'],
        how='inner'
    )

   
    df3 = pd.merge(df2, train, on='SamplingOperations_code', how='left')    
    # 5) test_df (filtrado por test_codes)
    col = 'IBD'   # columna en dfB que checas

    mask = df3[col].isna()          # True si NaN

    train_df = df3.loc[~mask]
    test_df = df3.loc[mask]  
    test_df=test_df.drop(columns=(['IBD', 'IBD_EQR', 'IBD_EQR_Status']))

    cleandf = clean_up(train= train_df,test=test_df)

    cleandf = cleaner(cleandf)

    return cleandf

In [142]:
# Ejemplo de uso:
region = 21
cleandf= build_region_train_test_epm(
    region=region,
    sites=sites,
    taxones=taxones,
    pressure=pressure,
    train=train,
    test_codes=test_codes,   # asegúrate de tenerlo definido
    fillna_with=0            # pon None si no quieres rellenar
)

Dropped exact duplicate columns: ['Pinrh02', 'Mean1Y_1770', 'Mean90Days_1647', 'Mean90Days_1648']
Dropped columns with >95% missing: ['Achaa01', 'Achac01', 'Achaf01', 'Achaf02', 'Achal01', 'Achan01', 'Achba01', 'Achbi02', 'Achca03', 'Achca04', 'Achco01', 'Achco02', 'Achcr01', 'Achde02', 'Achde03', 'Achdi01', 'Achdi02', 'Achdr01', 'Achel01', 'Achen01', 'Achex01', 'Achex02', 'Achfr01', 'Achfu01', 'Achge01', 'Achgr02', 'Achgr03', 'Achhe01', 'Achho01', 'Achhu01', 'Achja02', 'Achjo01', 'Achko01', 'Achku01', 'Achla03', 'Achla04', 'Achla06', 'Achle02', 'Achli01', 'Achli02', 'Achli03', 'Achlu02', 'Achma01', 'Achmi03', 'Achna02', 'Achne01', 'Achno01', 'Achpa01', 'Achpe02', 'Achpf01', 'Achps04', 'Achpu01', 'Achpy01', 'Achre01', 'Achro01', 'Achro02', 'Achru01', 'Achsa01', 'Achse02', 'Achsi01', 'Achst01', 'Achst03', 'Achsu06', 'Achsu07', 'Achte01', 'Achtr02', 'Achtr03', 'Achzh01', 'Achzi01', 'Actde01', 'Actno01', 'Adlba01', 'Adlbr01', 'Adlbr02', 'Adlmu02', 'Adlpa01', 'Adlsu01', 'Ampat01', 'Ampei01

KeyError: ('IBD', 'IBD_EQR', 'IBD_EQR_Status')

In [19]:
def crear_modelo(cleandf):
    cleandf = cleandf[cleandf['IBD'].notna()]
    X = cleandf.drop(columns=['IBD','IBD_EQR','IBD_EQR_Status'])
    y = cleandf['IBD']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    # 2) Identify column types
    num_cols = X.select_dtypes(include=['number']).columns
    cat_cols = X.columns.difference(num_cols)
    # 3) Preprocess
    # this preprocessor can handle missing values   
    pre = ColumnTransformer([
        # numerical features
        ('num', Pipeline([
            # imputation and scaling
            ('imp', SimpleImputer(strategy='median')),
            # scaling (RF doesn't need it, but other models might)
            # ('scaler', StandardScaler(with_mean=False))  # stays sparse with OHE
        ]), num_cols),
        # categorical features
        ('cat', Pipeline([
            # imputation as None being another category
            ('imp', SimpleImputer(strategy='constant', fill_value='None')),
            # one-hot encoding, ignoring unknown categories during inference
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols)

    ])
    # 4) Model
    clf = Pipeline([
        ('pre', pre),
        ('model', RandomForestRegressor(n_estimators=600, random_state=42, ))  # for regression
    ])  
    clf.fit(X_tr, y_tr)
    print("R2 train:", clf.score(X_tr, y_tr))
    print("R2 valid:", clf.score(X_te, y_te))

    return clf

In [20]:
clf = crear_modelo(cleandf)

R2 train: 0.9790034187053251
R2 valid: 0.8288111692690423


In [21]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def train_region_model(cleandf, target='IBD'):
    # 1) separar train y scoring dentro de la MISMA región
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # X / y
    drop_cols = [c for c in ['IBD','IBD_EQR','IBD_EQR_Status'] if c in cleandf.columns]
    X = df_train.drop(columns=drop_cols)
    y = df_train[target].astype(float)

    # 2) columnas num/cat
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    pre = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
        ]), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='(missing)')),
            ('ohe', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols)
    ])

    # 3) modelo
    clf = Pipeline([
        ('pre', pre),
        ('model', RandomForestRegressor(n_estimators=600, 
                                        random_state=42, 
                                        n_jobs=-1))
    ])

    # 4) hold-out (barajado dentro de la región)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    clf.fit(X_tr, y_tr)
    pred_tr = clf.predict(X_tr)
    pred_te = clf.predict(X_te)

    metrics = {
        'R2_train': r2_score(y_tr, pred_tr),
        'R2_valid': r2_score(y_te, pred_te),
        'MAE_train': mean_absolute_error(y_tr, pred_tr),
        'MAE_valid': mean_absolute_error(y_te, pred_te),
        'RMSE_train': mean_squared_error(y_tr, pred_tr),
        'RMSE_valid': mean_squared_error(y_te, pred_te),
    }

    # 5) (opcional) KFold simple dentro de la región
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_r2 = cross_val_score(clf, X, y, cv=kf, scoring='r2', n_jobs=-1)
    metrics['R2_CV_mean'] = cv_r2.mean()
    metrics['R2_CV_all']  = cv_r2

    # 6) predecir filas sin target de esta región
    if not df_score.empty:
        X_score = df_score.drop(columns=drop_cols, errors='ignore')
        preds_score = clf.predict(X_score)
        scored = df_score.copy()
        scored[target + '_pred'] = preds_score
    else:
        scored = pd.DataFrame(columns=cleandf.columns.tolist() + [target + '_pred'])

    return clf, metrics, scored


In [22]:
model, metrics, scored = train_region_model(cleandf, target='IBD')
print(metrics)
# scored trae las filas SIN IBD con la columna IBD_pred


{'R2_train': 0.9790856340204913, 'R2_valid': 0.8289758937124094, 'MAE_train': 0.27007747989276765, 'MAE_valid': 0.7792530335474669, 'RMSE_train': 0.14444726322609894, 'RMSE_valid': 1.2640867709374266, 'R2_CV_mean': np.float64(0.8384223048157503), 'R2_CV_all': array([0.82755961, 0.83745028, 0.83044998, 0.85299148, 0.84366017])}


In [54]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor


In [ ]:

def train_catboost_region(
    cleandf: pd.DataFrame,
    target: str = 'IBD',
    test_size: float = 0.20,
    random_state: int = 42,
    early_stopping_rounds: int = 200,
    cat_params: dict | None = None,
):
    """
    Entrena CatBoost para una región usando cleandf.
    - Separa train (con target) y score (sin target)
    - Preprocesa (imputación num/cat + OHE)
    - Entrena con early stopping
    - Regresa: modelo (Pipeline), métricas, scored_df (filas sin target con predicción)
    """
    # 1) separar train / score
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # X / y
    drop_cols = [c for c in ['IBD','IBD_EQR','IBD_EQR_Status'] if c in cleandf.columns]
    X = df_train.drop(columns=drop_cols, errors='ignore')
    y = df_train[target].astype(float)

    # split
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state)

    # 2) columnas num/cat
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    # preprocesamiento
    pre = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
        ]), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='(missing)')),
            ('ohe', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols)
    ])

    # 3) modelo CatBoost (parámetros por defecto + override opcional)
    base_params = dict(
        depth=6, learning_rate=0.05, n_estimators=3000,
        loss_function='RMSE', random_state=random_state, verbose=0
    )
    if cat_params:
        base_params.update(cat_params)

    cb = Pipeline([
        ('pre', pre),
        ('model', CatBoostRegressor(**base_params))
    ])

    # 4) ajustar: primero ajustamos el preprocesador para armar matrices, luego el modelo con early stopping
    Xtr_proc = pre.fit_transform(X_tr)
    Xte_proc = pre.transform(X_te)
    cb.named_steps['model'].fit(
        Xtr_proc, y_tr,
        eval_set=(Xte_proc, y_te),
        use_best_model=True,
        early_stopping_rounds=early_stopping_rounds
    )

    # 5) métricas en hold-out y train (usando el pipeline para que transforme igual)
    r2_tr  = cb.score(X_tr, y_tr)
    r2_te  = cb.score(X_te, y_te)
    pred_tr = cb.predict(X_tr); pred_te = cb.predict(X_te)
    metrics = {
        'R2_train': r2_tr,
        'R2_valid': r2_te,
        'MAE_train': mean_absolute_error(y_tr, pred_tr),
        'MAE_valid': mean_absolute_error(y_te, pred_te),
        'RMSE_train': mean_squared_error(y_tr, pred_tr),
        'RMSE_valid': mean_squared_error(y_te, pred_te),
        'best_iterations': int(cb.named_steps['model'].get_best_iteration() or base_params['n_estimators'])
    }

    # 6) predicciones para filas sin target de esta región (si existen)
    if not df_score.empty:
        X_score = df_score.drop(columns=drop_cols, errors='ignore')
        df_score[target + '_pred'] = cb.predict(X_score)
        scored_df = df_score
        
    else:
        scored_df = pd.DataFrame(columns=list(cleandf.columns) + [target + '_pred'])

    return cb, metrics, scored_df


In [24]:
model, metrics, scored = train_catboost_region(cleandf, target='IBD')
print(metrics)
# 'scored' contiene las filas sin IBD con la columna IBD_pred


{'R2_train': np.float64(0.9997330827914238), 'R2_valid': np.float64(0.9250414753976104), 'MAE_train': 0.03467258630740812, 'MAE_valid': 0.47896087936926185, 'RMSE_train': 0.0018434917092180043, 'RMSE_valid': 0.5540393186416179, 'best_iterations': 2997}


In [55]:
import os
import joblib
import numpy as np
import pandas as pd

# carpetas de salida
os.makedirs("models/regions", exist_ok=True)
os.makedirs("preds/regions", exist_ok=True)

all_metrics = []
all_scored  = []
models      = {}

for region in regiones:
    # --- construir el cleandf de la región ---
    out = build_region_train_test(
        region=region,
        sites=sites,
        taxones=taxones,
        pressure=pressure,
        train=train,
        test_codes=test_codes,
        fillna_with=0
    )
    # por si tu función devuelve varios objetos, nos quedamos con el DF limpio
    cleandf = out if isinstance(out, pd.DataFrame) else out[0]

    # --- entrenar modelo CatBoost por región ---
    model, metrics, scored = train_catboost_region(cleandf, target='IBD')

    # --- guardar modelo ---
    model_path = f"models/regions/catboost_region_{region}.joblib"
    joblib.dump(model, model_path)
    models[region] = model  # también en memoria

    # --- tamaños y features ---
    drop_cols = [c for c in ['IBD','IBD_EQR','IBD_EQR_Status'] if c in cleandf.columns]
    n_rows   = cleandf.shape[0]
    n_train  = cleandf['IBD'].notna().sum() if 'IBD' in cleandf.columns else np.nan
    n_score  = cleandf['IBD'].isna().sum() if 'IBD' in cleandf.columns else np.nan
    n_feats  = cleandf.drop(columns=drop_cols, errors='ignore').shape[1]

    # --- almacenar métricas ---
    row = {
        'region': region,
        'model_path': model_path,
        'n_rows': n_rows,
        'n_train': n_train,
        'n_to_score': n_score,
        'n_features': n_feats
    }
    row.update(metrics)  # R2_train, R2_valid, MAE, RMSE, best_iterations, etc.
    all_metrics.append(row)

    # --- guardar predicciones de la región (si hay filas sin IBD) ---
    if not scored.empty:
        scored['region'] = region
        scored_path = f"preds/regions/IBD_preds_region_{region}.parquet"
        scored.to_parquet(scored_path, index=False)
        all_scored.append(scored)

# --- DataFrames finales globales ---
results_df = pd.DataFrame(all_metrics).sort_values('region').reset_index(drop=True)
results_df.to_csv("models/region_metrics.csv", index=False)

preds_df = pd.concat(all_scored, ignore_index=True) if all_scored else pd.DataFrame()
if not preds_df.empty:
    preds_df.to_parquet("preds/IBD_preds_all_regions.parquet", index=False)

print("Guardados:")
print("  - métricas: models/region_metrics.csv")
print("  - modelos por región: models/regions/*.joblib")
print("  - predicciones por región: preds/regions/*.parquet")
if not preds_df.empty:
    print("  - predicciones globales: preds/IBD_preds_all_regions.parquet")


Dropped exact duplicate columns: ['Gompr03', 'Navkr01', 'Schex01']
Dropped columns with >95% missing: ['Achaf01', 'Achaf02', 'Achat01', 'Achca02', 'Achca04', 'Achch01', 'Achco01', 'Achda01', 'Achda02', 'Achde02', 'Achdr01', 'Achex01', 'Achex02', 'Achfu01', 'Achgr02', 'Achgr03', 'Achhi01', 'Achho01', 'Achho03', 'Achhu01', 'Achja02', 'Achko01', 'Achkr02', 'Achla03', 'Achla06', 'Achli03', 'Achob01', 'Achpf01', 'Achpu01', 'Achre01', 'Achro02', 'Achru01', 'Achth01', 'Achtr03', 'Achzh01', 'Adlbr02', 'Adlmi01', 'Adlsu01', 'Ampco01', 'Ampma01', 'Ampme01', 'Ampmi02', 'Ampmo01', 'Ampne01', 'Ampne02', 'Ampno01', 'Amppe01', 'Ampve01', 'Ampve02', 'Astfo01', 'Aulbr01', 'Auldi01', 'Aulps01', 'Aulsu01', 'Aulsu02', 'Bacpa01', 'Bacul01', 'Berru01', 'Bible01', 'Brane01', 'Brane02', 'Breke01', 'Calal01', 'Calfo01', 'Calsc01', 'Calsi01', 'Cocdi01', 'Cochu01', 'Cocne02', 'Cocne03', 'Cocpa01', 'Cocps02', 'Cocse01', 'Conwe01', 'Cosla02', 'Ctepu01', 'Cycat01', 'Cycco01', 'Cyccy01', 'Cycde02', 'Cycdi01', 'Cyckr

In [27]:
predicciones = []
all_metrics = {}  # guardará {region: métricas}

for region in regiones:
    cleandf = build_region_train_test(
        region=region,
        sites=sites,
        taxones=taxones,
        pressure=pressure,
        train=train,
        test_codes=test_codes,
        fillna_with=0
    )
    model, m, scored = train_catboost_region(cleandf, target='IBD')  # m = métricas de esa región
    all_metrics[region] = m

    # Mantener índice (SamplingOperations_code) y solo la predicción
    solo_ibd = scored[['IBD_pred']].copy()
    solo_ibd['region'] = region
    predicciones.append(solo_ibd)

predicciones = pd.concat(predicciones, axis=0)  # índice preservado
predicciones.index.name = 'SamplingOperations_code'
    

Dropped exact duplicate columns: ['Gompr03', 'Navkr01', 'Schex01']
Dropped columns with >95% missing: ['Achaf01', 'Achaf02', 'Achat01', 'Achca02', 'Achca04', 'Achch01', 'Achco01', 'Achda01', 'Achda02', 'Achde02', 'Achdr01', 'Achex01', 'Achex02', 'Achfu01', 'Achgr02', 'Achgr03', 'Achhi01', 'Achho01', 'Achho03', 'Achhu01', 'Achja02', 'Achko01', 'Achkr02', 'Achla03', 'Achla06', 'Achli03', 'Achob01', 'Achpf01', 'Achpu01', 'Achre01', 'Achro02', 'Achru01', 'Achth01', 'Achtr03', 'Achzh01', 'Adlbr02', 'Adlmi01', 'Adlsu01', 'Ampco01', 'Ampma01', 'Ampme01', 'Ampmi02', 'Ampmo01', 'Ampne01', 'Ampne02', 'Ampno01', 'Amppe01', 'Ampve01', 'Ampve02', 'Astfo01', 'Aulbr01', 'Auldi01', 'Aulps01', 'Aulsu01', 'Aulsu02', 'Bacpa01', 'Bacul01', 'Berru01', 'Bible01', 'Brane01', 'Brane02', 'Breke01', 'Calal01', 'Calfo01', 'Calsc01', 'Calsi01', 'Cocdi01', 'Cochu01', 'Cocne02', 'Cocne03', 'Cocpa01', 'Cocps02', 'Cocse01', 'Conwe01', 'Cosla02', 'Ctepu01', 'Cycat01', 'Cycco01', 'Cyccy01', 'Cycde02', 'Cycdi01', 'Cyckr

In [28]:
predicciones

,IBD_pred,region
SamplingOperations_code,,
S02000010_20080811,14.479493,18
S02000010_20100719,15.194359,18
S02000010_20150811,13.903476,18
S02000010_20160825,14.921369,18
S02000010_20170703,15.851655,18
...,...,...
S06700075_20120611,19.901358,2
S06700094_20210830,19.203143,2
S06700590_20210830,17.942659,2


In [29]:
metrics_df = (pd.DataFrame.from_dict(all_metrics, orient='index')
                .reset_index()
                .rename(columns={'index':'region'}))

In [30]:
metrics_df

,region,R2_train,R2_valid,MAE_train,MAE_valid,RMSE_train,RMSE_valid,best_iterations
0,18,0.999990,0.901993,0.005296,0.489012,4.207170e-05,0.538531,2995
1,5,0.999875,0.934270,0.022089,0.419943,7.775573e-04,0.471772,2998
2,4,1.000000,0.849195,0.000360,0.707182,1.976783e-07,1.160626,2997
3,10,0.999311,0.946923,0.059974,0.399498,5.664265e-03,0.426951,2999
4,22,1.000000,0.574058,0.000695,1.190135,6.443370e-07,3.580636,1036
5,9,0.993251,0.944179,0.111604,0.258880,2.223457e-02,0.180434,2997
6,21,0.999733,0.925041,0.034673,0.478961,1.843492e-03,0.554039,2997
7,20,1.000000,0.864287,0.000167,0.746150,3.870126e-08,0.860230,2992
8,12,0.998820,0.945756,0.062631,0.350562,6.375603e-03,0.306740,2997
9,8,0.999376,0.830042,0.041760,0.603596,2.618219e-03,0.740376,884


In [33]:
# tabla de regiones: columnas -> HERlvl1Code, HERlvl1Name
reg_tab = sites[['HERlvl1Code','HERlvl1Name']].drop_duplicates()

# aseguramos tipos
predicciones['region'] = predicciones['region'].astype(int)
reg_tab['HERlvl1Code'] = reg_tab['HERlvl1Code'].astype(int)

# diccionario y nueva columna
name_map = dict(zip(reg_tab['HERlvl1Code'], reg_tab['HERlvl1Name']))
predicciones['HERlvl1Name'] = predicciones['region'].map(name_map)
predicciones.drop(columns=['region'])


,IBD_pred,HERlvl1Name
SamplingOperations_code,,
S02000010_20080811,14.479493,ALSACE
S02000010_20100719,15.194359,ALSACE
S02000010_20150811,13.903476,ALSACE
S02000010_20160825,14.921369,ALSACE
S02000010_20170703,15.851655,ALSACE
...,...,...
S06700075_20120611,19.901358,ALPES INTERNES
S06700094_20210830,19.203143,ALPES INTERNES
S06700590_20210830,17.942659,ALPES INTERNES


In [56]:
# load ranges
    # save to csv
ranges = pd.read_csv("../../data/processed/ibd_eqr_ranges_by_herlvl1_continuous_midpoint.csv")
ranges.head(20)

,HERlvl1Name,IBD_EQR_Status,IBD_min,IBD_max,IBD_mid
0,ALPES INTERNES,Bad,0.000,9.800,9.30
1,ALPES INTERNES,Poor,9.800,13.225,10.30
2,ALPES INTERNES,Moderate,13.225,17.025,16.15
3,ALPES INTERNES,Good,17.025,18.725,17.90
4,ALPES INTERNES,High,18.725,20.000,19.55
5,ALSACE,Bad,0.000,6.925,5.40
6,ALSACE,Poor,6.925,10.425,8.45
7,ALSACE,Moderate,10.425,14.050,12.40
8,ALSACE,Good,14.050,17.125,15.70
9,ALSACE,High,17.125,20.000,18.55


In [37]:
def add_eqr_status(yhat: pd.DataFrame, ranges: pd.DataFrame) -> pd.DataFrame:
    """
    Adds the column 'IBD_EQR_Status_Predicted' to the `yhat` DataFrame by mapping the predicted IBD values 
    ('IBD_Predicted') into the appropriate bin defined by the [IBD_min, IBD_max) intervals in the `ranges` DataFrame 
    for the corresponding 'HERlvl1Name'. The topmost bin per region also includes its right endpoint.

    Parameters:
    ----------
    yhat : pd.DataFrame
        A DataFrame containing the predicted IBD values ('IBD_Predicted') and the corresponding 'HERlvl1Name'.
    ranges : pd.DataFrame
        A DataFrame containing the bin definitions for each 'HERlvl1Name', including columns:
        - 'HERlvl1Name': The region name.
        - 'IBD_EQR_Status': The status corresponding to the bin.
        - 'IBD_min': The lower bound of the bin (inclusive).
        - 'IBD_max': The upper bound of the bin (exclusive, except for the topmost bin).

    Returns:
    -------
    pd.DataFrame
        A copy of the `yhat` DataFrame with an additional column 'IBD_EQR_Status_Predicted', which contains the 
        mapped status for each prediction.

    Notes:
    -----
    - The function performs a cartesian merge between `yhat` and `ranges` based on 'HERlvl1Name'.
    - Each predicted value is matched to the bin where it falls within the [IBD_min, IBD_max) interval.
    - For the topmost bin in each region, the right endpoint (IBD_max) is included.
    - In case of ties (multiple bins matching a prediction), the first match is kept.
    """
    out = yhat.copy()
    out['__ix__'] = np.arange(len(out))

    # Copy ranges and calculate the maximum right endpoint for each region
    r = ranges[['HERlvl1Name', 'IBD_EQR_Status', 'IBD_min', 'IBD_max']].copy()
    r['__max_right__'] = r.groupby('HERlvl1Name')['IBD_max'].transform('max')

    # Cartesian merge by region, then keep the single interval that matches each prediction
    m = out.merge(r, on='HERlvl1Name', how='left')

    # Check if predictions fall within the bin intervals
    pred = m['IBD_pred'].astype(float)
    left_ok  = pred >= m['IBD_min']
    right_ok = (pred <  m['IBD_max']) | ((pred == m['IBD_max']) & (m['IBD_max'].eq(m['__max_right__'])))
    m = m[left_ok & right_ok]

    # In case of any ties, keep the first match; then map back to original rows
    m = m.sort_values(['__ix__', 'IBD_min', 'IBD_max']).drop_duplicates('__ix__', keep='first')
    status = m.set_index('__ix__')['IBD_EQR_Status']

    # Map the status back to the original DataFrame
    out['IBD_EQR_Status_Predicted'] = out['__ix__'].map(status)
    out = out.drop(columns='__ix__')
    return out

In [38]:
yhat2 = add_eqr_status(predicciones, ranges)
yhat2

,IBD_pred,region,HERlvl1Name,IBD_EQR_Status_Predicted
SamplingOperations_code,,,,
S02000010_20080811,14.479493,18,ALSACE,Good
S02000010_20100719,15.194359,18,ALSACE,Good
S02000010_20150811,13.903476,18,ALSACE,Moderate
S02000010_20160825,14.921369,18,ALSACE,Good
S02000010_20170703,15.851655,18,ALSACE,Good
...,...,...,...,...
S06700075_20120611,19.901358,2,ALPES INTERNES,High
S06700094_20210830,19.203143,2,ALPES INTERNES,High
S06700590_20210830,17.942659,2,ALPES INTERNES,Good


In [57]:
yhat2_clean = yhat2[yhat2['IBD_EQR_Status_Predicted'].notna()].copy()
len(yhat2_clean)

5412

In [59]:
dirty = yhat2[yhat2['IBD_EQR_Status_Predicted'].isna()].copy()
dirty

,IBD_pred,region,HERlvl1Name,IBD_EQR_Status_Predicted
SamplingOperations_code,,,,
S02025700_20160825,20.402804,18,ALSACE,NaN
S06000401_20200820,20.229204,5,JURA-PREALPES DU NORD,NaN
S06000401_20230717,20.048887,5,JURA-PREALPES DU NORD,NaN
S06000418_20200811,20.098163,5,JURA-PREALPES DU NORD,NaN
S06062400_20220302,20.048153,5,JURA-PREALPES DU NORD,NaN
...,...,...,...,...
S06159930_20200218,20.005038,2,ALPES INTERNES,NaN
S06159930_20230214,20.027815,2,ALPES INTERNES,NaN
S06592020_20120222,20.021544,2,ALPES INTERNES,NaN


In [60]:
np.min(dirty['IBD_pred'])

np.float64(20.002323952861655)

In [61]:
np.max(dirty['IBD_pred'])

np.float64(21.677480726550343)

In [62]:
yhat2['IBD_EQR_Status_Predicted'] = yhat2['IBD_EQR_Status_Predicted'].fillna('High')

In [65]:
import pandas as pd
fr_her1 = pd.read_csv("../../notebooks/04 random forest/un_lolazo_submission.csv")
fr_her1

,SamplingOperations_code,IBD_EQR_Status
0,S02000010_20080811,Moderate
1,S02000010_20100719,Good
2,S02000010_20150811,Moderate
3,S02000010_20170703,Good
4,S02000011_20100719,Good
...,...,...
5058,S06940940_20100708,Moderate
5059,S06940940_20230623,Moderate
5060,S06960950_20160629,High
5061,S06960950_20180719,High


In [66]:
# yhat2: index = SamplingOperations_code
# fr_her1: columna 'SamplingOperations_code'
innerjoin = (
    yhat2.reset_index()
         .merge(fr_her1, on='SamplingOperations_code', how='inner',
                suffixes=('_pred','_ref'))
)
innerjoin


,SamplingOperations_code,IBD_pred,region,HERlvl1Name,IBD_EQR_Status_Predicted,IBD_EQR_Status
0,S02000010_20080811,14.479493,18,ALSACE,Good,Moderate
1,S02000010_20100719,15.194359,18,ALSACE,Good,Good
2,S02000010_20150811,13.903476,18,ALSACE,Moderate,Moderate
3,S02000010_20170703,15.851655,18,ALSACE,Good,Good
4,S02000011_20100719,14.957075,18,ALSACE,Good,Good
...,...,...,...,...,...,...
5058,S06700075_20120611,19.901358,2,ALPES INTERNES,High,High
5059,S06700094_20210830,19.203143,2,ALPES INTERNES,High,Moderate
5060,S06700590_20210830,17.942659,2,ALPES INTERNES,Good,Good
5061,S06710014_20210823,19.735215,2,ALPES INTERNES,High,Good


In [69]:
count = 0
for r in innerjoin.itertuples(index=False):
    pred = str(r.IBD_EQR_Status_Predicted).strip().lower()
    tru  = str(r.IBD_EQR_Status).strip().lower()
    if pred == tru:
        count += 1

prop = count / len(innerjoin)
print(count, "/", len(innerjoin), "=>", prop)


4276 / 5063 => 0.8445585621173217


In [70]:
send_predictions = yhat2['IBD_EQR_Status_Predicted']

In [71]:
send_predictions

SamplingOperations_code
S02000010_20080811        Good
S02000010_20100719        Good
S02000010_20150811    Moderate
S02000010_20160825        Good
S02000010_20170703        Good
                        ...   
S06700075_20120611        High
S06700094_20210830        High
S06700590_20210830        Good
S06710014_20210823        High
S06820089_20120217        High
Name: IBD_EQR_Status_Predicted, Length: 5663, dtype: object

In [72]:
# change the name to "IBD_EQR_Status"
truesend = send_predictions.rename("IBD_EQR_Status")
truesend.to_csv("cat_boost_perh.csv")
truesend

SamplingOperations_code
S02000010_20080811        Good
S02000010_20100719        Good
S02000010_20150811    Moderate
S02000010_20160825        Good
S02000010_20170703        Good
                        ...   
S06700075_20120611        High
S06700094_20210830        High
S06700590_20210830        Good
S06710014_20210823        High
S06820089_20120217        High
Name: IBD_EQR_Status, Length: 5663, dtype: object